# 12 — Development-Selected Trading Strategy

This notebook selects a simple long-YES trading strategy using development
data only and evaluates the locked rule separately on the holdout and June
external blocks.

The weather model, continuous calibration and probability calibration remain
unchanged.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        manifest = (
            candidate
            / "data/manifests/"
            "12_trading_strategy_manifest.json"
        )

        if manifest.exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


ROOT = locate_repository(Path.cwd())

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "12_trading_strategy_manifest.json"
    ).read_text(encoding="utf-8")
)

candidates = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_development_trading_candidate_summary.csv"
)

locked_dates = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_locked_trading_date_panel.csv"
)

locked_trades = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_locked_trading_trade_panel.csv"
)

cost_sensitivity = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_locked_trading_cost_sensitivity.csv"
)

integrity = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_trading_strategy_integrity_checks.csv"
)

summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "12_trading_strategy_summary.csv"
)

print("Status:", manifest["status"])
print(
    "Balanced development dates:",
    manifest["balanced_development_dates"],
)
print(
    "Selected strategy:",
    manifest["selected_strategy"],
)
print(
    "Selected rule:",
    manifest["selected_rule"],
)
print(
    "Selected threshold:",
    manifest["selected_edge_threshold"],
)

Status: TRADING_STRATEGY_LOCKED_AND_EVALUATED
Balanced development dates: 29
Selected strategy: 24h_prior__tau_0.075
Selected rule: 24h_prior
Selected threshold: 0.075


## Trading rule

For event \(j\), settlement date \(d\) and decision rule \(r\), define the
estimated edge by

\[
e_{d,r,j}
=
\widehat p_{d,r,j}
-
p^{\mathrm{mkt}}_{d,r,j}.
\]

The strategy considers the event with the largest estimated edge for each
date. A long-YES position is entered only when

\[
e_{d,r,j}\geq \tau.
\]

For a one-share position and per-trade cost \(\kappa\), realised net payoff is

\[
\Pi_{d,r,j}
=
Y_{d,j}
-
p^{\mathrm{mkt}}_{d,r,j}
-
\kappa.
\]

At most one position is permitted for each settlement date.

In [2]:
selected = candidates.loc[
    candidates["selected_strategy"]
    .astype(str)
    .str.lower()
    .isin({"true", "1"})
]

assert len(selected) == 1

columns = [
    "candidate_id",
    "decision_rule",
    "edge_threshold",
    "trade_count",
    "mean_date_net_payoff",
    "standard_error_date_net_payoff",
    "within_one_standard_error",
    "strict_development_winner",
    "selected_strategy",
]

ordered = candidates.sort_values(
    [
        "selected_strategy",
        "strict_development_winner",
        "mean_date_net_payoff",
    ],
    ascending=[
        False,
        False,
        False,
    ],
)

print(
    ordered[
        columns
    ].head(15).to_string(
        index=False
    )
)

             candidate_id  decision_rule  edge_threshold  trade_count  mean_date_net_payoff  standard_error_date_net_payoff  within_one_standard_error  strict_development_winner  selected_strategy
     24h_prior__tau_0.075      24h_prior           0.075           25              0.120000                        0.083978                       True                      False               True
      6h_prior__tau_0.030       6h_prior           0.030           29              0.201569                        0.081607                       True                       True              False
      6h_prior__tau_0.000       6h_prior           0.000           29              0.201569                        0.081607                       True                      False              False
      6h_prior__tau_0.010       6h_prior           0.010           29              0.201569                        0.081607                       True                      False              False
      6h_prior_

## Chronological separation

Strategy selection uses only balanced development dates for which all four
decision rules have complete market and model probability books.

The selected decision rule and edge threshold are then locked. Neither the
10-date holdout nor the 30-date June block influences strategy selection.

In [3]:
assert (
    manifest["holdout_used_for_strategy_selection"]
    is False
)

assert (
    manifest["external_test_used_for_strategy_selection"]
    is False
)

assert (
    manifest["weather_model_reselected"]
    is False
)

assert (
    manifest["continuous_calibration_reselected"]
    is False
)

assert (
    manifest["probability_calibration_reselected"]
    is False
)

assert (
    manifest["strategy_refitted_before_external_test"]
    is False
)

print(
    "Holdout used for strategy selection:",
    False,
)

print(
    "June used for strategy selection:",
    False,
)

print(
    "Strategy refitted before June:",
    False,
)

Holdout used for strategy selection: False
June used for strategy selection: False
Strategy refitted before June: False


## Locked evaluation

Payoffs are reported separately for the holdout and June external blocks.
Dates without admissible market support remain part of the calendar-date
denominator and receive zero realised payoff.

In [4]:
print(
    summary.to_string(
        index=False
    )
)

assert set(
    summary["chronology_block"]
) == {
    "holdout",
    "external_test",
}

chronology_block  calendar_dates  market_supported_dates  market_unsupported_dates    selected_strategy selected_rule  edge_threshold  cost_per_trade  trade_count  trade_share_all_calendar_dates  cumulative_gross_payoff  cumulative_net_payoff  mean_calendar_date_net_payoff  standard_error_calendar_date_net_payoff  mean_supported_date_net_payoff  trade_win_share  maximum_drawdown_net_payoff
         holdout              10                      10                         0 24h_prior__tau_0.075     24h_prior           0.075            0.01            4                        0.400000                   0.6045                 0.5645                       0.056450                                 0.076965                        0.056450         0.500000                       -0.325
   external_test              30                      27                         3 24h_prior__tau_0.075     24h_prior           0.075            0.01           26                        0.866667                  -0

## Trading-cost sensitivity

The strategy is not reselected when trading costs change. The sensitivity
table applies alternative fixed costs to the same locked trades.

In [5]:
print(
    cost_sensitivity.sort_values(
        [
            "chronology_block",
            "cost_per_trade",
        ]
    ).to_string(
        index=False
    )
)

chronology_block    selected_strategy selected_rule  edge_threshold  cost_per_trade  calendar_dates  trade_count  cumulative_net_payoff  mean_calendar_date_net_payoff  standard_error_calendar_date_net_payoff
   external_test 24h_prior__tau_0.075     24h_prior           0.075           0.000              30           26                -0.3210                      -0.010700                                 0.047522
   external_test 24h_prior__tau_0.075     24h_prior           0.075           0.005              30           26                -0.4510                      -0.015033                                 0.047528
   external_test 24h_prior__tau_0.075     24h_prior           0.075           0.010              30           26                -0.5810                      -0.019367                                 0.047537
   external_test 24h_prior__tau_0.075     24h_prior           0.075           0.020              30           26                -0.8410                      -0.028033  

## Evidential limits

This is a reduced-form historical exercise. It does not model order-book
depth, spread crossing, partial fills, latency or market impact. Positive
historical payoff would therefore not establish executable profitability.

In [6]:
def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(
            {
                "true",
                "1",
                "yes",
                "y",
            }
        )
    )


assert as_bool(
    integrity["passed"]
).all()

trade_rows = locked_dates.loc[
    as_bool(
        locked_dates["trade"]
    )
]

assert not trade_rows[
    [
        "target_date",
        "chronology_block",
    ]
].duplicated().any()

assert (
    manifest[
        "maximum_positions_per_settlement_date"
    ]
    == 1
)

assert (
    manifest[
        "categorically_normalised_market_prices_used"
    ]
    is False
)

assert (
    manifest[
        "raw_market_prices_used"
    ]
    is True
)

print(
    "All Notebook 12 integrity checks passed:",
    True,
)

print(
    "Maximum positions per settlement date:",
    1,
)

print(
    "Raw market prices used:",
    True,
)

All Notebook 12 integrity checks passed: True
Maximum positions per settlement date: 1
Raw market prices used: True
